In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import os

# Vérification que TensorFlow voit bien ton environnement
print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


In [5]:
# Lecture du dossier Parquet généré par ton code Scala
path_data = "../data/dataframe"
df = pd.read_parquet(path_data)

print(f"✅ Données chargées : {len(df)} images.")
print(f"Labels trouvés : {df['label'].unique()}") # Doit être [1, 0]

✅ Données chargées : 14148 images.
Labels trouvés : [1 0]


In [9]:
import numpy as np

# Analyse de la structure réelle
df['pixel_count'] = df['image'].apply(lambda x: len(np.array(x).flatten()))
print("Statistiques des nombres de pixels par image :")
print(df['pixel_count'].value_counts())

# Verification de la premiere ligne pour voir a quoi ca ressemble
if len(df) > 0:
    sample = np.array(df['image'].iloc[0]).flatten()
    print(f"Exemple de contenu (taille {len(sample)}) : {sample[:10]}")

Statistiques des nombres de pixels par image :
pixel_count
32    9427
1     4721
Name: count, dtype: int64
Exemple de contenu (taille 32) : [array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
 array([1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 0.91686275,
        0.71088627, 0.9917098 , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        ])
 array([1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 0.97905098, 0.71911373,
        0.43898824, 0.99607843, 1.        , 1.        , 1.        ,
  

In [13]:
import numpy as np
from sklearn.model_selection import train_test_split

def convert_to_matrix(img_data):
    try:
        # On force la conversion de chaque sous-element en tableau numpy
        # pour obtenir une matrice 2D reelle (32, 32)
        matrix = np.array([np.array(x) for x in img_data])

        if matrix.shape == (32, 32):
            return matrix
        return None
    except:
        return None

# Nettoyage
print("Traitement et conversion des matrices...")
df['fixed_image'] = df['image'].apply(convert_to_matrix)
df_clean = df.dropna(subset=['fixed_image']).copy()

print(f"Images valides apres conversion : {len(df_clean)} sur {len(df)}")

if len(df_clean) == 0:
    # Diagnostic si cela echoue encore
    sample_val = df['image'].iloc[0]
    print(f"Echec. Format reel du premier element : {type(sample_val)}")
    if hasattr(sample_val, '__len__'):
        print(f"Nombre de sous-elements : {len(sample_val)}")
    raise ValueError("df_clean est vide. Verifiez la structure des donnees.")

# Preparation des tenseurs
X = np.stack(df_clean['fixed_image'].values)
X = X.reshape(-1, 32, 32, 1).astype('float32')
y = df_clean['label'].values

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Forme finale X_train : {X_train.shape}")

Traitement et conversion des matrices...
Images valides apres conversion : 9427 sur 14148
Forme finale X_train : (7541, 32, 32, 1)


In [14]:
model = models.Sequential([
    # Detection des formes de base
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 1)),
    layers.MaxPooling2D((2, 2)),

    # Detection de motifs plus complexes
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    # Sortie : 1 neurone avec sigmoid (0 = Rotten, 1 = Fresh selon ton Scala)
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("Modele pret pour l'entrainement.")

C:\Users\elyes\miniconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Modele pret pour l'entrainement.


In [15]:
print("Lancement de l'entrainement...")

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

print("Entrainement termine.")

Lancement de l'entrainement...
Epoch 1/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.7811 - loss: 0.4376 - val_accuracy: 0.8367 - val_loss: 0.3449
Epoch 2/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8417 - loss: 0.3331 - val_accuracy: 0.8298 - val_loss: 0.3628
Epoch 3/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8602 - loss: 0.3093 - val_accuracy: 0.8653 - val_loss: 0.2916
Epoch 4/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8666 - loss: 0.2870 - val_accuracy: 0.8659 - val_loss: 0.2954
Epoch 5/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8714 - loss: 0.2802 - val_accuracy: 0.8834 - val_loss: 0.2603
Epoch 6/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8822 - loss: 0.2523 - val_accuracy: 0.8796 - val_loss: 0.2670
Epoch 7/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8911 - loss: 0.2385 - val_accuracy: 0.8855 - val_loss: 0.2479
Epoch 8/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0

In [19]:
import sys
import os
import shutil

# 1. On vérifie que le dossier SavedModel existe bien
temp_export_dir = "temp_model_export"
if os.path.exists(temp_export_dir):
    print("Dossier SavedModel detecte. Lancement de la conversion...")

    # 2. Appel de la commande tf2onnx via le terminal
    # On utilise ! pour dire au notebook d'exécuter une commande système
    !{sys.executable} -m tf2onnx.convert --saved-model temp_model_export --output fruit_model.onnx --opset 13

    if os.path.exists("fruit_model.onnx"):
        print("Export ONNX reussi : fruit_model.onnx")
        # 3. Nettoyage (optionnel)
        # shutil.rmtree(temp_export_dir)
    else:
        print("Erreur : Le fichier ONNX n'a pas ete genere.")
else:
    print("Erreur : Dossier temp_model_export introuvable. Relance l'export du modele.")

Dossier SavedModel detecte. Lancement de la conversion...
Export ONNX reussi : fruit_model.onnx


I0000 00:00:1776611033.639638   44724 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776611038.975275   44724 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

2026-04-19 17:04:04,808 - WARNING - From C:\Users\elyes\miniconda3\Lib\site-packages\tf2onnx\tf_loader.py:190: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.

2026-04-19 17:04:04,821 - WARNING - '--tag' not specified for saved_model. Using --tag serve
2026-04-19 17:04:05,069 - INFO - Signatures found in model: [serve,serving_default].
2026-04-19 17:04:05,069 - WARNING - '--signatur